# Chocolate Sales - SQL analysis

In [4]:
import pandas as pd
import sqlite3

df = pd.read_csv('Chocolate Sales.csv')

data cleaning

In [5]:
df['Amount'] = df['Amount'].str.replace('[$,]', '', regex=True).astype(float)
df['Date']   = pd.to_datetime(df['Date'], dayfirst=True)
df

,Sales Person,Country,Product,Date,Amount,Boxes Shipped
0,Jehu Rudeforth,UK,Mint Chip Choco,2022-01-04,5320.00,180
1,Van Tuxwell,India,85% Dark Bars,2022-08-01,7896.00,94
2,Gigi Bohling,India,Peanut Butter Cubes,2022-07-07,4501.00,91
3,Jan Morforth,Australia,Peanut Butter Cubes,2022-04-27,12726.00,342
4,Jehu Rudeforth,UK,Peanut Butter Cubes,2022-02-24,13685.00,184
...,...,...,...,...,...,...
3277,Karlen McCaffrey,Australia,Spicy Special Slims,2024-05-17,5303.58,354
3278,Jehu Rudeforth,USA,White Choc,2024-06-07,7339.32,121
3279,Ches Bonnell,Canada,Organic Choco Syrup,2024-07-26,616.09,238
3280,Dotty Strutley,India,Eclairs,2024-07-28,2504.62,397


In [6]:
df.columns   = ['sales_person','country','product','date','amount','boxes_shipped']
df

,sales_person,country,product,date,amount,boxes_shipped
0,Jehu Rudeforth,UK,Mint Chip Choco,2022-01-04,5320.00,180
1,Van Tuxwell,India,85% Dark Bars,2022-08-01,7896.00,94
2,Gigi Bohling,India,Peanut Butter Cubes,2022-07-07,4501.00,91
3,Jan Morforth,Australia,Peanut Butter Cubes,2022-04-27,12726.00,342
4,Jehu Rudeforth,UK,Peanut Butter Cubes,2022-02-24,13685.00,184
...,...,...,...,...,...,...
3277,Karlen McCaffrey,Australia,Spicy Special Slims,2024-05-17,5303.58,354
3278,Jehu Rudeforth,USA,White Choc,2024-06-07,7339.32,121
3279,Ches Bonnell,Canada,Organic Choco Syrup,2024-07-26,616.09,238
3280,Dotty Strutley,India,Eclairs,2024-07-28,2504.62,397


make SQLoite database in memory

In [27]:
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')

3282

### Summary statistics

In [28]:
query = '''
SELECT
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount), 2) AS total_revenue,
    SUM(boxes_shipped) AS boxes_sold,
    ROUND(AVG(amount), 2) AS average_deal,
    ROUND(SUM(amount)/SUM(boxes_shipped), 2) AS revenue_per_box,
    COUNT(DISTINCT sales_person) AS sales_persons_amount,
    COUNT(DISTINCT country) AS countries,
    COUNT(DISTINCT product) AS products,
    DATE(MIN(date)) AS date_from,
    DATE(MAX(date)) AS date_to
FROM sales;
'''
pd.read_sql(query, conn)

,total_transactions,total_revenue,boxes_sold,average_deal,revenue_per_box,sales_persons_amount,countries,products,date_from,date_to
0,3282,19791571.86,540437,6030.34,36.62,25,6,22,2022-01-03,2024-08-31


### Revenue and sold boxes by country

In [31]:
query = '''
SELECT
    country,
    COUNT(*) AS transactions,
    SUM(amount) AS revenue,
    SUM(boxes_shipped) as boxes_shipped,
    ROUND(SUM(amount)/SUM(boxes_shipped), 2) AS revenue_per_box,
    ROUND(100*SUM(amount)/SUM(SUM(amount)) OVER (), 2) AS revenue_percent
FROM sales
GROUP BY country
ORDER BY revenue DESC;
'''
pd.read_sql(query, conn)

,country,transactions,revenue,boxes_shipped,revenue_per_box,revenue_percent
0,Australia,615,3646444.35,99618,36.60,18.42
1,UK,534,3365388.90,92523,36.37,17.00
2,India,552,3343730.83,89968,37.17,16.89
3,USA,537,3313858.09,81820,40.50,16.74
4,Canada,525,3078495.65,95158,32.35,15.55
5,New Zealand,519,3043654.04,81350,37.41,15.38


### Top ten sales persons

In [37]:
query = '''
SELECT 
    sales_person,
    COUNT(*) AS transactions,
    SUM(amount) AS revenue,
    AVG(amount) AS average_deal,
    SUM(boxes_shipped) AS total_boxes_sold,
    ROUND(SUM(amount)/SUM(boxes_shipped), 2) AS revenue_per_box
FROM sales
GROUP BY sales_person
ORDER BY revenue DESC
LIMIT 10;
'''
pd.read_sql(query, conn)

,sales_person,transactions,revenue,average_deal,total_boxes_sold,revenue_per_box
0,Ches Bonnell,144,1022599.96,7101.388611,23070,44.33
1,Oby Sorrel,147,1017204.12,6919.755918,26390,38.55
2,Madelene Upcott,135,1010028.72,7481.694222,22199,45.50
3,Kelci Walkden,162,1002929.10,6190.920370,26605,37.70
4,Brien Boise,159,997326.48,6272.493585,24738,40.32
5,Van Tuxwell,153,974425.09,6368.791438,20627,47.24
6,Dennison Crosswaite,147,931849.57,6339.112721,26862,34.69
7,Beverie Moffet,150,892421.37,5949.475800,28027,31.84
8,Kaine Padly,135,849062.76,6289.353778,22134,38.36
9,Marney O'Breen,135,836427.63,6195.760222,24595,34.01


### Top 10 products and their performance

In [40]:
query = '''
SELECT
    product,
    COUNT(*) AS transactions,
    ROUND(SUM(amount), 0) AS revenue,
    SUM(boxes_shipped) AS boxes,
    ROUND(SUM(amount)/SUM(boxes_shipped), 2) AS revenue_per_box,
    ROUND(100.0 * SUM(amount)/SUM(SUM(amount)) OVER (), 1) AS revenue_percent
FROM sales
GROUP BY product
ORDER BY revenue DESC
LIMIT 10
'''
pd.read_sql(query, conn)

,product,transactions,revenue,boxes,revenue_per_box,revenue_percent
0,Smooth Sliky Salty,177,1120201.0,26969,41.54,5.7
1,50% Dark Bites,180,1087659.0,29810,36.49,5.5
2,White Choc,174,1054257.0,25158,41.91,5.3
3,Peanut Butter Cubes,147,1036591.0,25339,40.91,5.2
4,Eclairs,180,996948.0,26678,37.37,5.0
5,99% Dark & Pure,147,960033.0,24818,38.68,4.9
6,85% Dark Bars,150,955268.0,23828,40.09,4.8
7,Organic Choco Syrup,156,945346.0,23602,40.05,4.8
8,Spicy Special Slims,162,938132.0,26662,35.19,4.7
9,Mint Chip Choco,135,904990.0,25149,35.99,4.6


### Sales by months

In [41]:
query = '''
SELECT
    STRFTIME('%Y-%m', date) AS month,
    COUNT(*) AS transactions,
    ROUND(SUM(amount), 0) AS revenue,
    SUM(boxes_shipped) AS boxes
FROM sales
GROUP BY month
ORDER BY month;
'''
pd.read_sql(query, conn)

,month,transactions,revenue,boxes
0,2022-01,154,896105.0,27535
1,2022-02,110,699377.0,18015
2,2022-03,131,749483.0,19561
3,2022-04,118,674051.0,21003
4,2022-05,135,752892.0,21856
5,2022-06,163,865144.0,26260
6,2022-07,149,803425.0,22876
7,2022-08,134,743148.0,19901
8,2023-01,154,958986.0,28189
9,2023-02,110,749617.0,18369


### Monthly growth

In [43]:
query = '''
WITH monthly_revenues AS (
    SELECT
        STRFTIME('%Y-%m', date) AS month,
        SUM(amount) AS revenue
    FROM sales
    GROUP BY month
)
SELECT
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
    ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month)) / LAG(revenue) OVER (ORDER BY month), 1) AS growth_percent
FROM monthly_revenues
ORDER BY month;
'''
pd.read_sql(query, conn)

,month,revenue,prev_month_revenue,growth_percent
0,2022-01,896105.00,NaN,NaN
1,2022-02,699377.00,896105.00,-22.0
2,2022-03,749483.00,699377.00,7.2
3,2022-04,674051.00,749483.00,-10.1
4,2022-05,752892.00,674051.00,11.7
5,2022-06,865144.00,752892.00,14.9
6,2022-07,803425.00,865144.00,-7.1
7,2022-08,743148.00,803425.00,-7.5
8,2023-01,958985.77,743148.00,29.0
9,2023-02,749617.46,958985.77,-21.8


### Assign categories to sales persons according to the sales they made

### Best selling products by country

In [51]:
query = '''
WITH revenues_by_product AS (
    SELECT 
        country, 
        product, 
        SUM(amount) AS revenue 
    FROM sales 
    GROUP BY country, product 
    ),
max_revenues AS (
    SELECT country, MAX(revenue) AS revenue 
    FROM revenues_by_product 
    GROUP BY country
    )

SELECT 
    revenues_by_product.country, 
    revenues_by_product.product, 
    max_revenues.revenue 
FROM revenues_by_product 
JOIN max_revenues 
    ON revenues_by_product.country = max_revenues.country 
    AND revenues_by_product.revenue = max_revenues.revenue
ORDER BY max_revenues.revenue DESC
'''
pd.read_sql(query, conn)

,country,product,revenue
0,Australia,50% Dark Bites,280066.82
1,New Zealand,Mint Chip Choco,277979.25
2,USA,Raspberry Choco,265127.63
3,India,Eclairs,252107.54
4,UK,Peanut Butter Cubes,252099.20
5,Canada,Smooth Sliky Salty,218122.40


In [52]:
query = '''
SELECT country, product, revenue
FROM (
    SELECT
        country,
        product,
        SUM(amount) AS revenue,
        RANK() OVER (
            PARTITION BY country
            ORDER BY SUM(amount) DESC
        ) AS r
    FROM sales
    GROUP BY country, product
)
WHERE r = 1
ORDER BY revenue DESC;
'''
pd.read_sql(query, conn)

,country,product,revenue
0,Australia,50% Dark Bites,280066.82
1,New Zealand,Mint Chip Choco,277979.25
2,USA,Raspberry Choco,265127.63
3,India,Eclairs,252107.54
4,UK,Peanut Butter Cubes,252099.20
5,Canada,Smooth Sliky Salty,218122.40


### Revenue by quarter

In [69]:
query = '''
SELECT
    STRFTIME('%Y', date) AS year,
    CAST((strftime('%m', date) + 2)/3 AS INTEGER) AS quarter,
    COUNT(*) AS transactions,
    SUM(amount) AS revenue,
    SUM(boxes_shipped) AS boxes
FROM sales
GROUP BY year, quarter
ORDER BY year, quarter;
'''
pd.read_sql(query, conn)

,year,quarter,transactions,revenue,boxes
0,2022,1,395,2344965.00,65111
1,2022,2,416,2292087.00,69119
2,2022,3,283,1546573.00,42777
3,2023,1,395,2516097.56,66599
4,2023,2,416,2472320.67,71066
5,2023,3,283,1654959.73,43746
6,2024,1,395,2644516.22,67002
7,2024,2,416,2587202.06,71050
8,2024,3,283,1732850.62,43967
